In [1]:
import random
import time
from datetime import datetime, timedelta
from pathlib import Path

import requests
import pandas as pd


# ------------------------
# 設定
# ------------------------
N_SELECT = 1000
DAYS = 90
VS_CURRENCY = "usd"

MIN_SLEEP_SEC = 2.3
JITTER_SEC = 0.7
MAX_RETRIES = 5
TIMEOUT_SEC = 30
RANDOM_SEED = 42

OUTPUT_DIR = Path(".")

COINS_LIST_URL = "https://api.coingecko.com/api/v3/coins/list"
MARKET_CHART_URL = "https://api.coingecko.com/api/v3/coins/{id}/market_chart"


# ------------------------
# セッション
# ------------------------
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})


def log(msg: str):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}", flush=True)


def fmt_time(seconds):
    return str(timedelta(seconds=int(seconds)))


def controlled_sleep():
    time.sleep(MIN_SLEEP_SEC + random.uniform(0, JITTER_SEC))


# ------------------------
# リトライ付きリクエスト
# ------------------------
def request_with_retry(url, params=None):
    wait = MIN_SLEEP_SEC

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = session.get(url, params=params, timeout=TIMEOUT_SEC)

            if r.status_code == 200:
                return r

            if r.status_code == 429:
                sleep_sec = wait * 2
                log(f"429 → {sleep_sec:.1f}s待機 ({attempt})")
                time.sleep(sleep_sec)
                wait *= 2
                continue

            if r.status_code in {500, 502, 503, 504}:
                sleep_sec = wait * 2
                log(f"{r.status_code} → {sleep_sec:.1f}s待機 ({attempt})")
                time.sleep(sleep_sec)
                wait *= 2
                continue

            r.raise_for_status()

        except requests.RequestException as e:
            if attempt == MAX_RETRIES:
                raise
            sleep_sec = wait * 2
            log(f"通信エラー → {sleep_sec:.1f}s待機 ({attempt})")
            time.sleep(sleep_sec)
            wait *= 2

    raise RuntimeError("retry失敗")


# ------------------------
# データ取得
# ------------------------
def fetch_coins_list():
    log("coins/list 取得中...")
    r = request_with_retry(COINS_LIST_URL)
    df = pd.DataFrame(r.json())
    df = df.dropna(subset=["id", "symbol", "name"])
    df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)
    log(f"coins数: {len(df)}")
    return df


def fetch_market_chart(coin_id):
    url = MARKET_CHART_URL.format(id=coin_id)
    params = {"vs_currency": VS_CURRENCY, "days": DAYS}

    r = request_with_retry(url, params=params)
    data = r.json()

    prices = data.get("prices", [])
    if not prices:
        return pd.DataFrame()

    df = pd.DataFrame(prices, columns=["timestamp", "price"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df = df.dropna().sort_values("timestamp")
    return df


# ------------------------
# メイン
# ------------------------
def main():
    random.seed(RANDOM_SEED)

    start_time = time.time()

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_csv = OUTPUT_DIR / f"coingecko_{N_SELECT}_{ts}.csv"

    coins = fetch_coins_list()

    sampled = coins.sample(n=N_SELECT, random_state=RANDOM_SEED).reset_index(drop=True)

    all_frames = []

    # ------------------------
    # 進捗ループ
    # ------------------------
    for i, row in sampled.iterrows():
        loop_start = time.time()

        coin_id = row["id"]
        symbol = row["symbol"]
        name = row["name"]

        try:
            df = fetch_market_chart(coin_id)

            if not df.empty:
                df["coin_id"] = coin_id
                df["symbol"] = symbol
                df["name"] = name
                all_frames.append(df)

        except Exception as e:
            log(f"{symbol} 失敗: {e}")

        # ------------------------
        # 進捗表示
        # ------------------------
        elapsed = time.time() - start_time
        done = i + 1
        rate = done / N_SELECT

        avg_time = elapsed / done
        remain = avg_time * (N_SELECT - done)

        log(
            f"[{done}/{N_SELECT} | {rate*100:.1f}%] {symbol} 完了\n"
            f"    経過: {fmt_time(elapsed)} / 残り: {fmt_time(remain)} / 平均: {avg_time:.2f}s/件"
        )

        controlled_sleep()

    # ------------------------
    # 保存
    # ------------------------
    if all_frames:
        result = pd.concat(all_frames)
        result.to_csv(out_csv, index=False, encoding="utf-8-sig")
        log(f"CSV保存完了: {out_csv}")

    total_time = time.time() - start_time
    log(f"総処理時間: {fmt_time(total_time)}")


if __name__ == "__main__":
    main()

[2026-04-11 07:19:15] coins/list 取得中...
[2026-04-11 07:19:16] coins数: 17707
[2026-04-11 07:19:17] [1/1000 | 0.1%] oza 完了
    経過: 0:00:01 / 残り: 0:30:26 / 平均: 1.83s/件
[2026-04-11 07:19:20] [2/1000 | 0.2%] fdai 完了
    経過: 0:00:04 / 残り: 0:40:01 / 平均: 2.41s/件
[2026-04-11 07:19:23] [3/1000 | 0.3%] energy 完了
    経過: 0:00:07 / 残り: 0:43:05 / 平均: 2.59s/件
[2026-04-11 07:19:27] [4/1000 | 0.4%] szar 完了
    経過: 0:00:11 / 残り: 0:47:35 / 平均: 2.87s/件
[2026-04-11 07:19:29] 429 → 4.6s待機 (1)
[2026-04-11 07:19:34] 429 → 9.2s待機 (2)
[2026-04-11 07:19:43] 429 → 18.4s待機 (3)
[2026-04-11 07:20:02] 429 → 36.8s待機 (4)
[2026-04-11 07:20:39] [5/1000 | 0.5%] ibtc 完了
    経過: 0:01:24 / 残り: 4:39:06 / 平均: 16.83s/件
[2026-04-11 07:20:43] [6/1000 | 0.6%] ionqon 完了
    経過: 0:01:27 / 残り: 4:01:16 / 平均: 14.56s/件
[2026-04-11 07:20:46] [7/1000 | 0.7%] tone 完了
    経過: 0:01:30 / 残り: 3:33:59 / 平均: 12.93s/件
[2026-04-11 07:20:50] [8/1000 | 0.8%] arg 完了
    経過: 0:01:34 / 残り: 3:15:27 / 平均: 11.82s/件
[2026-04-11 07:20:53] [9/1000 | 0.9%] ma